In [ ]:
import nvdlib
import json
import pandas as pd
import numpy as np
import re
import ast
from tqdm.notebook import tqdm
import sys
import string
import nltk
from nltk.tokenize import word_tokenize
import os
import fnmatch
import warnings
from pprint import pprint

from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')

stopwords = list(filter(lambda x: len(x)>1, stopwords.words('english')))
punctuation = set(string.punctuation)
punctuation.remove('(')
punctuation.remove(')')
punctuation.remove('$')

warnings.filterwarnings('ignore')

In [10]:
# Extensions to look at
LANGUAGE_FILE_EXTENSIONS = {
    'php': ['php', 'html', 'js'],
    'java': ['java', 'jsp', 'html', 'js']
}


def get_files(description: str) -> list[str]:
    extensions = LANGUAGE_FILE_EXTENSIONS.get(global_language.lower(), [])
    result = []
    for ext in extensions:
        result += re.findall(rf'\b\w+\.{ext}\b', description, re.IGNORECASE)
    return result

def get_versions(description: str) -> list[str]:
    result = re.findall(r'\d+\.\d+\.\d+', description)
    for version in result:
        description = description.replace(version, '')
    result += re.findall(r'\d+\.\d+', description)
    return result if result else ['0']

def get_vulnerability(description: str) -> str:
    desc = description.lower()
    if "sql" in desc:
        return "SQL Injection"
    elif any(x in desc for x in ["xss", "cross-site scripting", "cross site scripting"]):
        return "XSS"
    elif "file upload" in desc or "file inclusion" in desc:
        return "File Inclusion"
    elif "file access" in desc:
        return "File Access"
    elif "session" in desc:
        return "Session Fixation"
    elif "code injection" in desc:
        return "Code Injection"
    elif "command" in desc:
        return "Command Execution"
    elif "csrf" in desc or "request forgery" in desc:
        return "CSRF"
    return "NA"

def compare_versions(app_version: str, cve_versions: list[str]) -> bool:
    if not app_version or not cve_versions or cve_versions == ['0']:
        return True
    for cve_version in cve_versions:
        v1 = list(map(int, app_version.split('.')))
        v2 = list(map(int, cve_version.split('.')))
        size = min(len(v1), len(v2))
        if v2[:size] <= v1[:size]:
            return True
    return False

def get_parameters(description: str) -> list[str]:
    words = word_tokenize(description)
    words = [word for word in words if word.lower() not in stopwords.words('english') and word not in punctuation]

    params = [words[i-1] for i in range(1, len(words)) if words[i] in ["parameter", "parameters"]]
    params += [words[i+2] for i in range(1, len(words)-2) if (words[i-1]=='(' and words[i].isdigit() and words[i+1]==')')]
    params += [words[i+1] for i in range(len(words)-1) if words[i] in ["function", "$"]]

    exclusions = LANGUAGE_FILE_EXCLUSIONS.get(global_language.lower(), [])
    params = [param.replace('"', '') for param in params if all(ext not in param for ext in exclusions) and '/' not in param]
    return list(set(params))

def get_cve_from_navex(cve_df, file: str) -> str:
    for _, row in cve_df.iterrows():
        if any(file_name in file for file_name in row['filenames']):
            return row['id']
    return 'NA'

def filter_on_files(file_names: list[str], directory_path: str) -> bool:
    if not file_names:
        return True
    for root, _, files in os.walk(directory_path):
        for file_name in file_names:
            if any(fnmatch.fnmatch(f, file_name) for f in files):
                return True
    return False

def substr_as_identifier(main_string: str, substring: str) -> bool:
    pattern = re.compile(rf'{re.escape(substring)}(?![a-zA-Z0-9])')
    return bool(pattern.search(main_string))

def get_files_from_parameters(parameters: list[str], directory_path: str) -> list[str]:
    if not parameters:
        return []
    file_paths = []
    for root, _, files in os.walk(directory_path):
        for filename in files:
            file_path = os.path.join(root, filename)
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    if any(substr_as_identifier(content, param) for param in parameters):
                        file_paths.append(file_path)
            except Exception:
                continue
    return list(set(file_paths))

In [24]:
tqdm.pandas()
# Global language setting
global_language = 'java'
directory_path = "/path/to/app_direc"
appName =""
extension = ''
appVersion = '2.9.3'

In [25]:
r = nvdlib.searchCVE(keywordSearch=appName)
jsonFormattedCVE = json.dumps(ast.literal_eval(str(r)))
cve_ld = pd.read_json(jsonFormattedCVE)

In [76]:
cve=cve_ld.copy()

In [78]:
cve = pd.read_json(jsonFormattedCVE)

cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: list(filter(lambda x: x["lang"]=="en", descriptions)))
cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: descriptions[0]['value'])
cve['versions'] = cve['descriptions'].apply(getVersions)

cve['filenames'] = cve['descriptions'].apply(getFiles)
cve['cve_vulnerability'] = cve['descriptions'].apply(getVulnerability)
cve['parameters'] = cve['descriptions'].apply(getParameters)
cve['relevant_version'] = cve['versions'].apply(lambda x: compareVersions(appVersion, x))# and compareVersions(x[0], ['6.0']))

cve = cve[cve['cve_vulnerability']!='NA']
cve = cve[cve['relevant_version']==True]

cve = cve[cve['filenames'].apply(filterOnFiles)]
cve = cve[cve['parameters'].progress_apply(lambda params: getFilesFromParameters(params) != [])]
cve = cve[cve['filenames'].map(lambda x: x!=[]) | cve['parameters'].map(lambda x: x!=[])]
cve['filenames'] = cve.progress_apply(lambda x: x.filenames if x.filenames!=[] else list(map(lambda x: x.split('/')[-1], getFilesFromParameters(x.parameters))), axis=1)
cve = cve[cve['parameters'].apply(lambda params: params != ["query"] and isinstance(params, list))]
cve = cve.loc[:, ['id', 'cve_vulnerability', 'versions', 'filenames', 'parameters', 'descriptions']]

cve.to_excel(f"{appName}-{extension}cve.xlsx")
cve.shape

(1, 6)

In [ ]:
# Output of the navex joern extension
appName = 'tika'
navex = pd.read_json(f'paths/{appName}-{extension}output.json')


In [45]:
import pandas as pd

# Define variables
output_file = f'paths/{appName}-{extension}filtered.json'


# Filter rows with pathid >= 12
filtered_navex = navex[navex["pathid"] >= 12]

# Save the filtered DataFrame back to JSON
filtered_navex.to_json(output_file, orient='records', lines=True)


In [46]:
filtered_data = filtered_navex.to_dict(orient='records')  # Convert DataFrame back to list of dictionaries

# Save filtered data to JSON file
with open(output_file, 'w') as file:
    json.dump(filtered_data, file, indent=4)

In [48]:
import ast
import numpy as np

def getCVEidsFromPath(row, pathRow):
    """
    Match CVE identifiers based on filenames and parameters.

    Args:
        row (dict): CVE row with 'filenames', 'parameters', 'cve_vulnerability', and 'id'.
        pathRow (dict): Path row with 'filename', 'vulnerability', 'code', and 'methodname'.

    Returns:
        str or NaN: CVE ID if a match is found, otherwise NaN.
    """
    flag = False
    cve_id = np.nan

    # Parse 'filenames' and 'parameters' if stored as strings
    files = row['filenames']
    if type(files) != list:
        files = ast.literal_eval(files)

    cveParams = row['parameters']
    if type(cveParams) != list:
        cveParams = ast.literal_eval(cveParams)

    # Default to empty list if no files or parameters
    if not files:
        files = ['']
    if not cveParams:
        cveParams = []

    # Iterate over filenames
    for fileName in files:
        if row['cve_vulnerability'] == pathRow['vulnerability']:
            # Check if fileName matches pathRow's filename
            if not cveParams and fileName.lower() in pathRow['filename'].lower():
                flag = True
            else:
                # Check parameters in code or as method name
                for param in cveParams:
                    if (
                        fileName.lower() in pathRow['filename'].lower() and 
                        (substrAsIdentifier(pathRow['code'], param) or param == pathRow['methodname'])
                    ):
                        flag = True
            if flag:
                cve_id = row['id']
                break  # Stop once a match is found

    return cve_id

In [41]:
CVE_ids = navex.progress_apply(lambda navexRow: list(cve.apply(lambda x: getCVEidsFromPath(x, navexRow), axis=1).dropna().unique()), axis=1)
navex['CVE_ids'] = CVE_ids
navex.head() 

  0%|          | 0/40793 [00:00<?, ?it/s]

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids
0,1,SQL Injection,275249,invoke,tomcat55/container/catalina/src/share/org/apac...,214,request.getRequest(),FALSE,[]
1,1,SQL Injection,236989,doFilter,tomcat55/container/catalina/src/share/org/apac...,160,ServletRequest request,FALSE,[]
2,1,SQL Injection,237067,doFilter,tomcat55/container/catalina/src/share/org/apac...,188,request,FALSE,[]
3,1,SQL Injection,237073,internalDoFilter,tomcat55/container/catalina/src/share/org/apac...,192,ServletRequest request,FALSE,[]
4,1,SQL Injection,237280,internalDoFilter,tomcat55/container/catalina/src/share/org/apac...,252,request,FALSE,[]


In [51]:
# Check rows that matched with a CVE
emptyList = pd.Series([np.nan] * len(navex['CVE_ids'])).fillna('[]')
navex[navex['CVE_ids'].astype(str) != emptyList].explode('CVE_ids').head()

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids
142,1,XSS,352831,getValue,tomcat55/container/catalina/src/share/org/apac...,1122,this.getAttribute(name),FALSE,CVE-2008-1947
143,1,XSS,352830,,tomcat55/container/catalina/src/share/org/apac...,1122,return (getAttribute(name));,NA,CVE-2008-1947
144,1,XSS,352827,,tomcat55/container/catalina/src/share/org/apac...,1120,public Object getValue(String name),NA,CVE-2007-3384
144,1,XSS,352827,,tomcat55/container/catalina/src/share/org/apac...,1120,public Object getValue(String name),NA,CVE-2008-1947
145,1,XSS,568719,newInstance,tomcat55/container/modules/storeconfig-ha/src/...,118,attributes.getValue(attr),FALSE,CVE-2007-3384


In [52]:
identified=navex[navex['CVE_ids'].astype(str) != emptyList].explode('CVE_ids')

In [54]:
uknow_paths=list(set(navex["pathid"].unique()).difference(set(identified["pathid"].unique())))

In [56]:
path_tocheck=[]
for id_path in uknow_paths:
    path_tocheck.append(list(navex["code"][navex["pathid"]==id_path]))

In [57]:
import json


# Convert the list of lists to JSON format
paths_json = json.dumps(path_tocheck, indent=4)

# Save the JSON to a file
with open('paths.json', 'w') as json_file:
    json_file.write(paths_json)


In [58]:
navex = navex.explode('CVE_ids')
navex.shape

(42064, 9)

In [59]:
navex = navex.merge(cve, how='left', left_on='CVE_ids', right_on='id').drop(columns=['id', 'cve_vulnerability'])

In [ ]:
matchedCVEs = {e for e in navex['CVE_ids'].dropna()}
print("Number of exploit matches:", len(matchedCVEs), "out of #" + str(len(cve['id'])), "CVEs")
pprint(matchedCVEs)

In [471]:
# Save data to excel
dfToSave = navex.dropna()
dfToSave.reset_index(inplace=True)
dfToSave.to_excel(f"cve/{appName}-{extension}navex.xlsx")
dfToSave.shape

(14936, 14)

In [20]:
xssPaths = navex[navex['vulnerability'] == 'XSS']
# xssPaths = navex
idsAcrossDb = xssPaths[xssPaths.apply(lambda x: 'query' in x['code'], 1)]['pathid'].unique()
acrossDb = xssPaths[xssPaths.apply(lambda x: x['pathid'] in idsAcrossDb, 1)]
notDb = xssPaths[xssPaths['pathid'].apply(lambda x: x not in acrossDb['pathid'])]
set(acrossDb['CVE_ids'].unique()) - set(notDb['CVE_ids'].unique())

set()

In [23]:
navex

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids,versions,filenames,parameters,descriptions
0,1,SQL Injection,443908,getName,liquibase-master/liquibase-standard/src/main/j...,25,"this.getAttribute(""name"", String.class)",FALSE,NaN,NaN,NaN,NaN,NaN
1,1,SQL Injection,443907,,liquibase-master/liquibase-standard/src/main/j...,25,"return getAttribute(""name"", String.class);",NA,NaN,NaN,NaN,NaN,NaN
2,1,SQL Injection,443905,,liquibase-master/liquibase-standard/src/main/j...,23,public String getName(),NA,NaN,NaN,NaN,NaN,NaN
3,1,SQL Injection,353710,get,liquibase-master/liquibase-standard/src/main/j...,239,getName(),FALSE,NaN,NaN,NaN,NaN,NaN
4,1,SQL Injection,353706,get,liquibase-master/liquibase-standard/src/main/j...,239,(LiquibaseSerializable) object.getSerializable...,FALSE,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5958,333,XSS,256907,alreadyExists,liquibase-master/liquibase-standard/src/main/j...,131,backingIndex.getName(),FALSE,CVE-2020-2283,[1.4.5],[],[],Jenkins Liquibase Runner Plugin 1.4.5 and earl...
5959,333,XSS,441361,<init>,liquibase-master/liquibase-standard/src/main/j...,31,String indexName,FALSE,CVE-2020-2283,[1.4.5],[],[],Jenkins Liquibase Runner Plugin 1.4.5 and earl...
5960,333,XSS,441368,<init>,liquibase-master/liquibase-standard/src/main/j...,33,indexName,FALSE,CVE-2020-2283,[1.4.5],[],[],Jenkins Liquibase Runner Plugin 1.4.5 and earl...
5961,333,XSS,441427,setName,liquibase-master/liquibase-standard/src/main/j...,55,String name,FALSE,CVE-2020-2283,[1.4.5],[],[],Jenkins Liquibase Runner Plugin 1.4.5 and earl...
